# DreamerV3: JAX vs PyTorch — Paper Benchmark

**Purpose:** Publication-quality benchmark comparing JAX and PyTorch DreamerV3
implementations with real HM3D ObjectNav observations.

**Design:**
1. Collect real Habitat observations into a shared replay buffer
2. Benchmark training steps from the same buffer for both frameworks
3. Measure env stepping time separately
4. 3 seeds × 200 timed steps per framework

**Output:** LaTeX table + PDF bar chart for thesis methodology section.

**Note on PyTorch baseline:** We compare against NM512/dreamerv3-torch, the most
widely-used PyTorch DreamerV3 implementation. r2dreamer modifies the architecture
(removes encoder), so was excluded for a fair framework-level comparison.

```bash
srun --partition=dev_h100 --gres=gpu:1 --time=00:30:00 uv run jupyter nbconvert --to notebook --execute notebooks/dreamerv3_jax_vs_pytorch_paper.ipynb --inplace
```

In [1]:
import os, sys, time, subprocess
from pathlib import Path
from datetime import date

import numpy as np
import matplotlib
matplotlib.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.dpi': 150,
})
import matplotlib.pyplot as plt
import wandb

project_root = Path(os.path.abspath("")).parent if Path(os.path.abspath("")).name == "notebooks" else Path(os.path.abspath(""))
os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")

# Ensure dreamerv3-torch is cloned
torch_dreamer_path = project_root / "external" / "dreamerv3-torch"
if not torch_dreamer_path.exists():
    torch_dreamer_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/NM512/dreamerv3-torch.git",
                    str(torch_dreamer_path)], check=True)

# Benchmark parameters
OBS_SHAPE_CHW = (3, 64, 64)
OBS_SHAPE_HWC = (64, 64, 3)
NUM_ACTIONS = 4
BATCH_SIZE = 16
SEQ_LEN = 50
HIDDEN_SIZE = 512
LATENT_CLASSES = 32
LATENT_DIMS = 32
WARMUP_STEPS = 5
BENCH_STEPS = 200
SEEDS = [0, 1, 2]
PREFILL_STEPS = 2000  # Collect real Habitat obs

print(f"Benchmark: {BENCH_STEPS} steps x {len(SEEDS)} seeds, batch={BATCH_SIZE}, seq_len={SEQ_LEN}")
print(f"Obs: {OBS_SHAPE_CHW}, Actions: {NUM_ACTIONS}, Prefill: {PREFILL_STEPS}")

Working directory: /pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA
Benchmark: 200 steps x 3 seeds, batch=16, seq_len=50
Obs: (3, 64, 64), Actions: 4, Prefill: 2000


## 1. Collect Real Habitat Observations

In [2]:
from src.dreamerv3.configs import DreamerConfig
from src.dreamerv3.replay_buffer import ReplayBuffer
from src.dreamerv3.env_habitat import HabitatObjectNavEnv

collect_config = DreamerConfig(
    obs_shape=OBS_SHAPE_CHW,
    max_episode_steps=200,
    split="val_mini",
    buffer_capacity=PREFILL_STEPS + 1000,
)

env = HabitatObjectNavEnv(collect_config, max_geodesic=5.0)

# Also measure env stepping time
print(f"Collecting {PREFILL_STEPS} real Habitat observations...")
env_times = []
buffer = ReplayBuffer(collect_config)
obs = env.reset()
for i in range(PREFILL_STEPS):
    action = np.random.randint(0, NUM_ACTIONS)
    t0 = time.perf_counter()
    next_obs = env.step(action)
    env_times.append(time.perf_counter() - t0)
    buffer.add(obs["image"], action, next_obs["reward"], next_obs["done"])
    obs = next_obs if not next_obs["done"] else env.reset()

env.close()

env_mean = np.mean(env_times)
env_std = np.std(env_times)
print(f"Buffer size: {buffer.size}")
print(f"Env step time: {env_mean:.4f} ± {env_std:.4f} s")
print(f"Env steps/sec: {1/env_mean:.1f}")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


/pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring


2026-03-26 08:13:29,632 Initializing dataset ObjectNav-v1


2026-03-26 08:13:29,848 initializing sim Sim-v0


...

PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
[08:14:34:480638]:[Warning]:[Scene] SemanticScene.h(331)::checkFileExists : ::loadSemanticSceneDescriptor: File `data/scene_datasets/hm3d_v0.2/minival/00802-wcojb4TFT35/wcojb4TFT35.basis.scn` does not exist.  Aborting load.
[08:14:34:480684]:[Warning]:[Scene] SemanticScene.cpp(124)::loadSemanticSceneDescriptor : SSD File Naming Issue! Neither SemanticAttributes-provided name : `data/scene_datasets/hm3d_v0.2/minival/00802-wcojb4TFT35/wcojb4TFT35.basis.scn` nor constructed filename : `data/scene_datasets/hm3d_v0.2/minival/00802-wcojb4TFT35/info_semantic.j

PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
[08:14:35:990720]:[Warning]:[Scene] SemanticScene.h(331)::checkFileExists : ::loadSemanticSceneDescriptor: File `data/scene_datasets/hm3d_v0.2/minival/00800-TEEsavR23oF/TEEsavR23oF.basis.scn` does not exist.  Aborting load.
[08:14:35:990764]:[Warning]:[Scene] SemanticScene.cpp(124)::loadSemanticSceneDescriptor : SSD File Naming Issue! Neither SemanticAttributes-provided name : `data/scene_datasets/hm3d_v0.2/minival/00800-TEEsavR23oF/TEEsavR23oF.basis.scn` nor constructed filename : `data/scene_datasets/hm3d_v0.2/minival/00800-TEEsavR23oF/info_semantic.j

Renderer: NVIDIA H100/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 570.195.03
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-broken-buffer-dsa
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-framebuffer-invalidation-wants-draw-binding
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-

    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-broken-buffer-dsa
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-framebuffer-invalidation-wants-draw-binding
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
Buffer size: 2000
Env step time: 0.0018 ± 0.0008 s
Env steps/sec: 563.9


GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring
[08:14:36:980457]:[Warning]:[Scene] SemanticScene.h(331)::checkFileExists : ::loadSemanticSceneDescriptor: File `data/scene_datasets/hm3d_v0.2/minival/00802-wcojb4TFT35/wcojb4TFT35.basis.scn` does not exist.  Aborting load.
[08:14:36:980502]:[Warning]:[Scene] SemanticScene.cpp(124)::loadSemanticSceneDescriptor : SSD File Naming Issue! Neither SemanticAttributes-provided name : `data/scene_datasets/hm3d_v0.2/minival/00802-wcojb4TFT35/wcojb4TFT35.basis.scn` nor constructed filename : `data/scene_datasets/hm3d_v0.2/minival/00802-wcojb4TFT35/info_semantic.json` exist on disk.
[08:14:36:980517]:[Error]:[Scene] SemanticScene.cpp(140)::loadSemanticSceneDescriptor : SSD Load Failur

## 2. JAX Benchmark (3 seeds)

In [3]:
import jax
import jax.numpy as jnp
from src.dreamerv3.agent import DreamerAgent

print(f"JAX backend: {jax.default_backend()}, devices: {jax.devices()}")

jax_config = DreamerConfig(
    obs_shape=OBS_SHAPE_CHW,
    num_actions=NUM_ACTIONS,
    batch_size=BATCH_SIZE,
    seq_len=SEQ_LEN,
    hidden_size=HIDDEN_SIZE,
    latent_classes=LATENT_CLASSES,
    latent_dims=LATENT_DIMS,
    buffer_capacity=PREFILL_STEPS + 1000,
)

jax_all_times = []  # list of lists, one per seed
jax_all_mem = []

for seed in SEEDS:
    rng_key = jax.random.PRNGKey(seed)
    rng_key, init_key = jax.random.split(rng_key)
    agent = DreamerAgent(jax_config, init_key)

    # Warmup
    for _ in range(WARMUP_STEPS):
        rng_key, train_key = jax.random.split(rng_key)
        batch = buffer.sample(BATCH_SIZE, SEQ_LEN)
        metrics = agent.train_step(batch, train_key)
    
    # Timed
    seed_times = []
    for i in range(BENCH_STEPS):
        rng_key, train_key = jax.random.split(rng_key)
        batch = buffer.sample(BATCH_SIZE, SEQ_LEN)
        jax.block_until_ready(agent.wm_state.params)
        t0 = time.perf_counter()
        metrics = agent.train_step(batch, train_key)
        jax.block_until_ready(metrics)
        t1 = time.perf_counter()
        seed_times.append(t1 - t0)

    try:
        mem_gb = jax.devices()[0].memory_stats()["peak_bytes_in_use"] / 1e9
    except Exception:
        mem_gb = float("nan")

    jax_all_times.append(seed_times)
    jax_all_mem.append(mem_gb)
    print(f"  JAX seed={seed}: {np.mean(seed_times):.4f} ± {np.std(seed_times):.4f} s/step, mem={mem_gb:.2f} GB")

jax_flat = [t for seed_times in jax_all_times for t in seed_times]
jax_mean = np.mean(jax_flat)
jax_std = np.std(jax_flat)
jax_mem = np.mean(jax_all_mem)
print(f"\nJAX overall: {jax_mean:.4f} ± {jax_std:.4f} s/step, mem={jax_mem:.2f} GB")

JAX backend: gpu, devices: [CudaDevice(id=0)]


  JAX seed=0: 0.0399 ± 0.0003 s/step, mem=5.82 GB


  JAX seed=1: 0.0413 ± 0.0025 s/step, mem=5.82 GB


  JAX seed=2: 0.0406 ± 0.0010 s/step, mem=5.82 GB

JAX overall: 0.0406 ± 0.0017 s/step, mem=5.82 GB


## 3. PyTorch Benchmark (3 seeds)

In [4]:
import torch
import argparse
import gym

sys.path.insert(0, str(torch_dreamer_path))
from ruamel.yaml import YAML
import models as torch_models

# Load and configure
yaml = YAML(typ="safe")
with open(torch_dreamer_path / "configs.yaml") as f:
    all_configs = yaml.load(f)
defaults = all_configs.get("defaults", {})
defaults["batch_size"] = BATCH_SIZE
defaults["batch_length"] = SEQ_LEN
defaults["size"] = [OBS_SHAPE_HWC[0], OBS_SHAPE_HWC[1]]
defaults["num_actions"] = NUM_ACTIONS
defaults["dyn_deter"] = HIDDEN_SIZE
defaults["dyn_hidden"] = HIDDEN_SIZE
defaults["dyn_stoch"] = LATENT_CLASSES
defaults["dyn_discrete"] = LATENT_DIMS
defaults["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"
defaults["precision"] = 32
torch_config = argparse.Namespace(**defaults)

obs_space = gym.spaces.Dict({
    "image": gym.spaces.Box(0, 255, OBS_SHAPE_HWC, dtype=np.uint8),
})
act_space = gym.spaces.Box(0, 1, (NUM_ACTIONS,), dtype=np.float32)
act_space.discrete = True

print(f"PyTorch device: {torch_config.device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch device: cuda:0
GPU: NVIDIA H100


In [5]:
def make_torch_batch_from_buffer():
    """Sample from shared buffer, convert to NM512 format (numpy, HWC)."""
    jax_batch = buffer.sample(BATCH_SIZE, SEQ_LEN)
    # Convert CHW float32 [0,1] -> HWC uint8 [0,255]
    obs_chw = np.array(jax_batch["obs"])  # (B, T, 3, H, W) float32
    obs_hwc = np.transpose(obs_chw, (0, 1, 3, 4, 2))  # (B, T, H, W, 3)
    obs_uint8 = (obs_hwc * 255).astype(np.uint8)
    actions_idx = np.array(jax_batch["actions"])  # (B, T) int32
    actions_onehot = np.eye(NUM_ACTIONS, dtype=np.float32)[actions_idx]  # (B, T, 4)
    return {
        "image": obs_uint8,
        "action": actions_onehot,
        "reward": np.array(jax_batch["rewards"]),
        "is_first": np.array(jax_batch["is_first"]),
        "is_terminal": np.zeros_like(np.array(jax_batch["dones"])),
        "discount": np.ones_like(np.array(jax_batch["dones"])),
    }

def torch_train_step(wm, beh, batch):
    """One full training step: world model + behavior."""
    post, context, wm_metrics = wm._train(batch)
    objective = lambda feat, state, action: wm.heads["reward"](
        wm.dynamics.get_feat(state)
    ).mode()
    beh_metrics = beh._train(post, objective)
    return wm_metrics, beh_metrics

torch_all_times = []
torch_all_mem = []

for seed in SEEDS:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    wm = torch_models.WorldModel(obs_space, act_space, 0, torch_config).to(torch_config.device)
    beh = torch_models.ImagBehavior(torch_config, wm).to(torch_config.device)

    # Warmup
    for _ in range(WARMUP_STEPS):
        batch = make_torch_batch_from_buffer()
        torch_train_step(wm, beh, batch)

    # Timed
    seed_times = []
    for i in range(BENCH_STEPS):
        batch = make_torch_batch_from_buffer()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        torch_train_step(wm, beh, batch)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        seed_times.append(t1 - t0)

    if torch.cuda.is_available():
        mem_gb = torch.cuda.max_memory_allocated() / 1e9
    else:
        mem_gb = float("nan")

    torch_all_times.append(seed_times)
    torch_all_mem.append(mem_gb)
    print(f"  PyTorch seed={seed}: {np.mean(seed_times):.4f} ± {np.std(seed_times):.4f} s/step, mem={mem_gb:.2f} GB")

    del wm, beh  # Free GPU memory between seeds
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

torch_flat = [t for seed_times in torch_all_times for t in seed_times]
torch_mean = np.mean(torch_flat)
torch_std = np.std(torch_flat)
torch_mem = np.mean(torch_all_mem)
print(f"\nPyTorch overall: {torch_mean:.4f} ± {torch_std:.4f} s/step, mem={torch_mem:.2f} GB")

Encoder CNN shapes: {'image': (64, 64, 3)}
Encoder MLP shapes: {}
Decoder CNN shapes: {'image': (64, 64, 3)}
Decoder MLP shapes: {}


/pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/external/dreamerv3-torch/tools.py:747: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Optimizer model_opt has 15686787 variables.
Optimizer actor_opt has 1054728 variables.
Optimizer value_opt has 1181439 variables.


/pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/external/dreamerv3-torch/models.py:116: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._use_amp):


/pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/external/dreamerv3-torch/models.py:160: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._use_amp):
/pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/external/dreamerv3-torch/models.py:299: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._use_amp):
/pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/external/dreamerv3-torch/models.py:323: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._use_amp):


  PyTorch seed=0: 0.2603 ± 0.0039 s/step, mem=3.55 GB
Encoder CNN shapes: {'image': (64, 64, 3)}
Encoder MLP shapes: {}
Decoder CNN shapes: {'image': (64, 64, 3)}
Decoder MLP shapes: {}
Optimizer model_opt has 15686787 variables.
Optimizer actor_opt has 1054728 variables.
Optimizer value_opt has 1181439 variables.


  PyTorch seed=1: 0.2600 ± 0.0031 s/step, mem=3.55 GB
Encoder CNN shapes: {'image': (64, 64, 3)}
Encoder MLP shapes: {}
Decoder CNN shapes: {'image': (64, 64, 3)}
Decoder MLP shapes: {}
Optimizer model_opt has 15686787 variables.
Optimizer actor_opt has 1054728 variables.
Optimizer value_opt has 1181439 variables.


  PyTorch seed=2: 0.2609 ± 0.0356 s/step, mem=3.55 GB

PyTorch overall: 0.2604 ± 0.0208 s/step, mem=3.55 GB


## 4. Results

In [6]:
# Compute per-seed means for error bars
jax_seed_means = [np.mean(t) for t in jax_all_times]
torch_seed_means = [np.mean(t) for t in torch_all_times]
jax_seed_std = np.std(jax_seed_means)
torch_seed_std = np.std(torch_seed_means)
speedup = torch_mean / jax_mean

# WandB logging
today = date.today().strftime("%Y%m%d")
wandb.init(
    project="dreamerv3-objectnav",
    name=f"paper-benchmark-{today}",
    tags=["paper-benchmark", "jax", "pytorch"],
    group="paper-jax-vs-pytorch",
    config={
        "obs_shape": OBS_SHAPE_CHW,
        "num_actions": NUM_ACTIONS,
        "batch_size": BATCH_SIZE,
        "seq_len": SEQ_LEN,
        "hidden_size": HIDDEN_SIZE,
        "warmup_steps": WARMUP_STEPS,
        "bench_steps": BENCH_STEPS,
        "seeds": SEEDS,
        "prefill_steps": PREFILL_STEPS,
        "gpu": "NVIDIA H100",
        "real_habitat_obs": True,
    },
)
wandb.log({
    "jax/mean_step_time": jax_mean,
    "jax/std_step_time": jax_std,
    "jax/peak_gpu_gb": jax_mem,
    "pytorch/mean_step_time": torch_mean,
    "pytorch/std_step_time": torch_std,
    "pytorch/peak_gpu_gb": torch_mem,
    "env/mean_step_time": env_mean,
    "env/std_step_time": env_std,
    "speedup": speedup,
})

# Summary table
print("\n" + "=" * 62)
print(f"{'Metric':<35} {'JAX':>10} {'PyTorch':>10}")
print("-" * 62)
print(f"{'Train step time (s)':<35} {jax_mean:>10.4f} {torch_mean:>10.4f}")
print(f"{'  ± std (s)':<35} {jax_std:>10.4f} {torch_std:>10.4f}")
print(f"{'  ± across seeds (s)':<35} {jax_seed_std:>10.4f} {torch_seed_std:>10.4f}")
print(f"{'Peak GPU memory (GB)':<35} {jax_mem:>10.2f} {torch_mem:>10.2f}")
print(f"{'Steps/second':<35} {1/jax_mean:>10.1f} {1/torch_mean:>10.1f}")
print("-" * 62)
print(f"{'Env step time (s)':<35} {env_mean:>10.4f}")
print(f"{'Total step time (train+env) (s)':<35} {jax_mean+env_mean:>10.4f} {torch_mean+env_mean:>10.4f}")
print(f"{'End-to-end speedup':<35} {(torch_mean+env_mean)/(jax_mean+env_mean):>10.2f}x")
print("-" * 62)
print(f"{'Training speedup (JAX/PyTorch)':<35} {speedup:>10.2f}x")
print(f"{'Seeds':<35} {len(SEEDS):>10}")
print(f"{'Steps per seed':<35} {BENCH_STEPS:>10}")
print("=" * 62)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ul/ul_student/ul_hfj15/.netrc.


wandb: Currently logged in as: sailer-luca (sailer-luca-university-ulm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run e9zzl9ej


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /pfs/data6/home/ul/ul_student/ul_hfj15/Master-Thesis-3D-VLA/wandb/run-20260326_082211-e9zzl9ej
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run paper-benchmark-20260326


wandb: ⭐️ View project at https://wandb.ai/sailer-luca-university-ulm/dreamerv3-objectnav


wandb: 🚀 View run at https://wandb.ai/sailer-luca-university-ulm/dreamerv3-objectnav/runs/e9zzl9ej



Metric                                     JAX    PyTorch
--------------------------------------------------------------
Train step time (s)                     0.0406     0.2604
  ± std (s)                             0.0017     0.0208
  ± across seeds (s)                    0.0006     0.0004
Peak GPU memory (GB)                      5.82       3.55
Steps/second                              24.7        3.8
--------------------------------------------------------------
Env step time (s)                       0.0018
Total step time (train+env) (s)         0.0423     0.2622
End-to-end speedup                        6.19x
--------------------------------------------------------------
Training speedup (JAX/PyTorch)            6.42x
Seeds                                        3
Steps per seed                             200


## 5. LaTeX Table

In [7]:
e2e_jax = jax_mean + env_mean
e2e_torch = torch_mean + env_mean
e2e_speedup = e2e_torch / e2e_jax

latex = r"""\begin{table}[h]
\centering
\caption{DreamerV3 training performance: JAX vs.\ PyTorch on NVIDIA H100.
Measured over """ + str(BENCH_STEPS) + r""" training steps $\times$ """ + str(len(SEEDS)) + r""" seeds
with real HM3D ObjectNav observations (64$\times$64).
PyTorch baseline: NM512/dreamerv3-torch.}
\label{tab:jax-vs-pytorch}
\begin{tabular}{lcc}
\toprule
\textbf{Metric} & \textbf{JAX} & \textbf{PyTorch} \\
\midrule
""" + f"""Training step time (s) & ${jax_mean:.4f} \pm {jax_std:.4f}$ & ${torch_mean:.4f} \pm {torch_std:.4f}$ \\\\
Peak GPU memory (GB) & ${jax_mem:.2f}$ & ${torch_mem:.2f}$ \\\\
Training speedup & \multicolumn{{2}}{{c}}{{${speedup:.1f}\times$ (JAX)}} \\\\
\midrule
Env step time (s) & \multicolumn{{2}}{{c}}{{${env_mean:.4f} \pm {env_std:.4f}$}} \\\\
End-to-end step time (s) & ${e2e_jax:.4f}$ & ${e2e_torch:.4f}$ \\\\
End-to-end speedup & \multicolumn{{2}}{{c}}{{${e2e_speedup:.1f}\times$ (JAX)}} \\\\
""" + r"""\bottomrule
\end{tabular}
\end{table}"""

print(latex)

# Save to file
os.makedirs("output", exist_ok=True)
with open("output/jax_vs_pytorch_table.tex", "w") as f:
    f.write(latex)
print("\nSaved to output/jax_vs_pytorch_table.tex")

\begin{table}[h]
\centering
\caption{DreamerV3 training performance: JAX vs.\ PyTorch on NVIDIA H100.
Measured over 200 training steps $\times$ 3 seeds
with real HM3D ObjectNav observations (64$\times$64).
PyTorch baseline: NM512/dreamerv3-torch.}
\label{tab:jax-vs-pytorch}
\begin{tabular}{lcc}
\toprule
\textbf{Metric} & \textbf{JAX} & \textbf{PyTorch} \\
\midrule
Training step time (s) & $0.0406 \pm 0.0017$ & $0.2604 \pm 0.0208$ \\
Peak GPU memory (GB) & $5.82$ & $3.55$ \\
Training speedup & \multicolumn{2}{c}{$6.4	imes$ (JAX)} \\
\midrule
Env step time (s) & \multicolumn{2}{c}{$0.0018 \pm 0.0008$} \\
End-to-end step time (s) & $0.0423$ & $0.2622$ \\
End-to-end speedup & \multicolumn{2}{c}{$6.2	imes$ (JAX)} \\
\bottomrule
\end{tabular}
\end{table}

Saved to output/jax_vs_pytorch_table.tex


<>:23: SyntaxWarning: invalid escape sequence '\p'
<>:23: SyntaxWarning: invalid escape sequence '\p'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:23: SyntaxWarning: invalid escape sequence '\p'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:23: SyntaxWarning: invalid escape sequence '\p'
<>:23: SyntaxWarning: invalid escape sequence '\p'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:24: SyntaxWarning: invalid escape sequence '\m'
<>:23: SyntaxWarning: invalid escape sequence '\p'
<>:24: SyntaxWarning: invalid escape sequence '\m'
/scratch/slurm_tmpdir/job_3769607/ipykernel_1625451/4191479753.py:23: SyntaxWarning: invalid escape sequence '\p'
  """ + r"""\bottomrule
/scratch/slurm_tmpdir/job_3769607/ipykernel_1625451/4191479753.py:23: SyntaxWarning: invalid escape sequence '\p'
  """ + r"""\bottomrule
/scratch/slurm_tmpdir/job_3769607/ipykernel_1625451/4191479753.py:24: SyntaxWarning: invalid escape sequence '\m

## 6. Publication Figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

colors = {"JAX": "#2171b5", "PyTorch": "#cb181d"}

# --- Panel A: Training step time ---
ax = axes[0]
x = [0, 1]
means = [jax_mean, torch_mean]
# Error bars: std across seed means (between-seed variability)
errs = [jax_seed_std, torch_seed_std]
bars = ax.bar(x, means, yerr=errs, capsize=5, width=0.5,
              color=[colors["JAX"], colors["PyTorch"]], edgecolor="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(["JAX", "PyTorch"])
ax.set_ylabel("Time per training step (s)")
ax.set_title("(a) Training Step Time")
# Extend y-axis to make room for labels
ax.set_ylim(0, max(means) * 1.25)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(errs) + max(means)*0.03,
            f"{val:.3f}s", ha="center", va="bottom", fontsize=10)
# Add speedup annotation
ax.annotate(f"{speedup:.1f}x", xy=(0.5, max(means)*0.45), fontsize=14,
            ha="center", fontweight="bold", color="#333333")

# --- Panel B: GPU Memory ---
ax = axes[1]
mem_vals = [jax_mem, torch_mem]
bars = ax.bar(x, mem_vals, width=0.5,
              color=[colors["JAX"], colors["PyTorch"]], edgecolor="black", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(["JAX", "PyTorch"])
ax.set_ylabel("Peak GPU Memory (GB)")
ax.set_title("(b) GPU Memory Usage")
ax.set_ylim(0, max(mem_vals) * 1.25)
for bar, val in zip(bars, mem_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(mem_vals)*0.03,
            f"{val:.1f}", ha="center", va="bottom", fontsize=10)

plt.suptitle("DreamerV3 Framework Comparison (NVIDIA H100, HM3D 64x64)",
             y=1.03, fontsize=13, fontweight="bold")
plt.tight_layout()

# Save as PDF (vector) and PNG
fig.savefig("output/jax_vs_pytorch_benchmark.pdf", bbox_inches="tight", dpi=300)
fig.savefig("output/jax_vs_pytorch_benchmark.png", bbox_inches="tight", dpi=300)
wandb.log({"paper_figure": wandb.Image(fig)})
plt.show()

print("Saved to output/jax_vs_pytorch_benchmark.pdf")

wandb.finish()
print("Benchmark complete.")